# Обучение RuBERT для детекции фейковых новостей

Файн-тюнинг `DeepPavlov/rubert-base-cased` на русскоязычном датасете.  
Классификация: **0 = fake**, **1 = real**.

## Настройка рабочей директории

Переходим в корень проекта и проверяем наличие датасета.

In [ ]:
import os

os.chdir("C:/Users/pozoy/Desktop/MISIS/final")
print("Рабочая директория:", os.getcwd())
print("Датасет найден:", os.path.exists("../../data/ready_dataset.csv"))

## Импорты

Загрузка всех необходимых библиотек.

In [ ]:
import json
import random
from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd
import torch
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
)
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    get_linear_schedule_with_warmup,
)
from tqdm.auto import tqdm

## Конфигурация

Все гиперпараметры вынесены в один датакласс `Config` — менять нужно только здесь.  
Новые параметры по сравнению с исходной версией:
- `label_smoothing` — снижает уверенность модели и помогает против переобучения.
- `patience` — количество эпох без улучшения `val_f1` до остановки (early stopping).
- `use_amp` — mixed precision: автоматически включается при наличии GPU.

In [ ]:
@dataclass
class Config:
    data_path: str = "../../data/ready_dataset.csv"
    text_column: str = "combined_text"
    label_column: str = "label"

    model_name: str = "DeepPavlov/rubert-base-cased"
    output_dir: str = "../../models/rubert"

    max_length: int = 256
    batch_size: int = 8
    epochs: int = 4
    learning_rate: float = 2e-5
    weight_decay: float = 0.01
    warmup_ratio: float = 0.1
    grad_clip: float = 1.0
    label_smoothing: float = 0.1  # сглаживание меток против переобучения
    patience: int = 2  # early stopping: эпох без улучшения val_f1

    test_size: float = 0.1
    val_size: float = 0.1
    random_state: int = 42
    num_workers: int = 0

    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    use_amp: bool = torch.cuda.is_available()  # mixed precision только на GPU


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)

## Загрузка и разбивка данных

Читаем `data/ready_dataset.csv`, очищаем пропуски и делим на train / val / test (80 / 10 / 10)  
со стратификацией по метке.

In [ ]:
def load_dataset(config: Config):
    df = pd.read_csv(config.data_path)

    for col in [config.text_column, config.label_column]:
        if col not in df.columns:
            raise ValueError(f"Column '{col}' not found. Available: {list(df.columns)}")

    df = df[[config.text_column, config.label_column]].copy()
    df = df.dropna()
    df[config.text_column] = df[config.text_column].astype(str).str.strip()
    df = df[df[config.text_column] != ""]

    df[config.label_column] = pd.to_numeric(df[config.label_column], errors="coerce")
    df = df.dropna(subset=[config.label_column])
    df[config.label_column] = df[config.label_column].astype(int)

    labels = sorted(df[config.label_column].unique().tolist())
    if labels != [0, 1]:
        raise ValueError(f"Expected labels [0, 1], got {labels}")

    train_val_df, test_df = train_test_split(
        df,
        test_size=config.test_size,
        random_state=config.random_state,
        stratify=df[config.label_column],
    )

    # val_size задан относительно всего датасета — пересчитываем для оставшейся части
    relative_val_size = config.val_size / (1 - config.test_size)

    train_df, val_df = train_test_split(
        train_val_df,
        test_size=relative_val_size,
        random_state=config.random_state,
        stratify=train_val_df[config.label_column],
    )

    return (
        train_df.reset_index(drop=True),
        val_df.reset_index(drop=True),
        test_df.reset_index(drop=True),
    )

## Dataset класс

`NewsDataset` возвращает токенизированные примеры **без паддинга**.  
Паддинг до максимальной длины текущего батча добавляется динамически через `DataCollatorWithPadding` —  
это экономит память и ускоряет обучение на коротких текстах.

In [ ]:
class NewsDataset(Dataset):
    def __init__(self, df, tokenizer, text_column, label_column, max_length):
        self.texts = df[text_column].tolist()
        self.labels = df[label_column].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        # padding намеренно не задаётся — динамический паддинг через DataCollatorWithPadding
        encoded = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_length,
        )
        encoded["labels"] = self.labels[idx]
        return encoded

## DataLoader'ы

`DataCollatorWithPadding` дополняет каждый батч до длины самого длинного примера в нём —  
вместо фиксированного `max_length=256` для всех примеров.

In [ ]:
def create_dataloaders(train_df, val_df, test_df, tokenizer, config: Config):
    kwargs = dict(
        text_column=config.text_column,
        label_column=config.label_column,
        max_length=config.max_length,
    )
    train_dataset = NewsDataset(train_df, tokenizer, **kwargs)
    val_dataset = NewsDataset(val_df, tokenizer, **kwargs)
    test_dataset = NewsDataset(test_df, tokenizer, **kwargs)

    collator = DataCollatorWithPadding(tokenizer)

    train_loader = DataLoader(
        train_dataset,
        batch_size=config.batch_size,
        shuffle=True,
        num_workers=config.num_workers,
        collate_fn=collator,
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=config.batch_size,
        shuffle=False,
        num_workers=config.num_workers,
        collate_fn=collator,
    )
    test_loader = DataLoader(
        test_dataset,
        batch_size=config.batch_size,
        shuffle=False,
        num_workers=config.num_workers,
        collate_fn=collator,
    )

    return train_loader, val_loader, test_loader

## Метрики

Считаем accuracy, F1, precision и recall по классу **1 (real)**.

In [ ]:
def calculate_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, pos_label=1, zero_division=0),
        "precision": precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        "recall": recall_score(y_true, y_pred, pos_label=1, zero_division=0),
    }

## Обучение (одна эпоха)

`train_epoch` обучает модель за одну эпоху. Улучшения по сравнению с исходной версией:

- **Label smoothing** — вместо `outputs.loss` используем `CrossEntropyLoss(label_smoothing=0.1)`,  
  что снижает уверенность модели и уменьшает переобучение.
- **Mixed precision (AMP)** — `torch.cuda.amp.autocast` + `GradScaler` ускоряют обучение на GPU  
  за счёт вычислений в `float16`; на CPU автоматически отключается.
- **Gradient clipping** — стабилизирует обучение при больших градиентах.

In [ ]:
def train_epoch(model, dataloader, optimizer, scheduler, config: Config):
    model.train()
    total_loss = 0.0
    device = config.device
    loss_fct = torch.nn.CrossEntropyLoss(label_smoothing=config.label_smoothing)
    scaler = torch.cuda.amp.GradScaler(enabled=config.use_amp)

    for batch in tqdm(dataloader, desc="Train", leave=False):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        with torch.cuda.amp.autocast(enabled=config.use_amp):
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = loss_fct(outputs.logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), config.grad_clip)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        total_loss += loss.item()

    return total_loss / max(len(dataloader), 1)

## Оценка модели

Вычисляет loss и метрики на переданном DataLoader'е (val или test) без обновления весов.  
При оценке label smoothing **не применяется** — считаем честный cross-entropy loss.

In [ ]:
@torch.no_grad()
def evaluate(model, dataloader, device):
    model.eval()
    total_loss = 0.0
    y_true, y_pred = [], []
    loss_fct = torch.nn.CrossEntropyLoss()  # без label smoothing при оценке

    for batch in tqdm(dataloader, desc="Eval", leave=False):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = loss_fct(outputs.logits, labels)
        preds = torch.argmax(outputs.logits, dim=1)

        total_loss += loss.item()
        y_true.extend(labels.cpu().tolist())
        y_pred.extend(preds.cpu().tolist())

    metrics = calculate_metrics(y_true, y_pred)
    metrics["loss"] = total_loss / max(len(dataloader), 1)
    metrics["y_true"] = y_true
    metrics["y_pred"] = y_pred
    return metrics

## Сохранение артефактов

Сохраняет в `models/rubert/` следующие файлы (имена сохранены):

| Файл | Содержимое |
|---|---|
| `config.json` | параметры обучения |
| `history.json` | лосс и метрики по эпохам |
| `metrics.json` | итоговые метрики (загружается Streamlit-приложением) |
| `best_val_metrics.json` | метрики лучшей эпохи на валидации |
| `test_predictions.csv` | предсказания модели на тесте |

In [ ]:
def save_json(path, data):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=4)


def save_training_artifacts(
    config: Config, history, best_epoch, best_val_metrics, test_metrics
):
    ensure_dir(config.output_dir)

    results = {
        "rubert": {
            "val_acc": best_val_metrics["accuracy"],
            "val_f1": best_val_metrics["f1"],
            "test_acc": test_metrics["accuracy"],
            "test_f1": test_metrics["f1"],
            "test_precision": test_metrics["precision"],
            "test_recall": test_metrics["recall"],
            "best_epoch": best_epoch,
            "best_params": {
                "model_name": config.model_name,
                "max_length": config.max_length,
                "batch_size": config.batch_size,
                "epochs": config.epochs,
                "learning_rate": config.learning_rate,
                "weight_decay": config.weight_decay,
            },
        }
    }

    save_json(os.path.join(config.output_dir, "config.json"), asdict(config))
    save_json(os.path.join(config.output_dir, "history.json"), history)
    save_json(os.path.join(config.output_dir, "metrics.json"), results)

    pd.DataFrame(
        {
            "y_true": test_metrics["y_true"],
            "y_pred": test_metrics["y_pred"],
        }
    ).to_csv(
        os.path.join(config.output_dir, "test_predictions.csv"),
        index=False,
        encoding="utf-8",
    )

## Основная функция обучения

`train_rubert` запускает полный цикл:

1. Загружает данные и инициализирует модель.
2. Обучает эпохами, сохраняя лучший чекпоинт по `val_f1` в `models/rubert/best_model/`.
3. **Early stopping** — останавливается, если `val_f1` не улучшается `patience` эпох подряд.
4. Загружает лучший чекпоинт и оценивает на тестовой выборке.

In [ ]:
def train_rubert(config: Config):
    set_seed(config.random_state)
    ensure_dir(config.output_dir)

    print("=== Обучение RuBERT ===")
    print(f"Устройство:      {config.device}  (AMP: {config.use_amp})")
    print(f"Label smoothing: {config.label_smoothing}  |  patience: {config.patience}")
    print("Разметка классов: 0 = fake, 1 = real")

    train_df, val_df, test_df = load_dataset(config)
    print(f"\nTrain: {len(train_df)}  |  Val: {len(val_df)}  |  Test: {len(test_df)}")

    tokenizer = AutoTokenizer.from_pretrained(config.model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        config.model_name, num_labels=2
    ).to(config.device)

    train_loader, val_loader, test_loader = create_dataloaders(
        train_df, val_df, test_df, tokenizer, config
    )

    total_steps = len(train_loader) * config.epochs
    warmup_steps = int(total_steps * config.warmup_ratio)
    optimizer = AdamW(
        model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay
    )
    scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

    history = []
    best_val_f1 = -1.0
    best_epoch = -1
    best_val_metrics = None
    epochs_no_improve = 0
    best_model_dir = os.path.join(config.output_dir, "best_model")

    for epoch in range(1, config.epochs + 1):
        print(f"\nEpoch {epoch}/{config.epochs}")

        train_loss = train_epoch(model, train_loader, optimizer, scheduler, config)
        val_metrics = evaluate(model, val_loader, config.device)

        history.append(
            {
                "epoch": epoch,
                "train_loss": train_loss,
                "val_loss": val_metrics["loss"],
                "val_acc": val_metrics["accuracy"],
                "val_f1": val_metrics["f1"],
                "val_precision": val_metrics["precision"],
                "val_recall": val_metrics["recall"],
            }
        )

        print(
            f"train_loss={train_loss:.4f} | val_loss={val_metrics['loss']:.4f} | "
            f"val_acc={val_metrics['accuracy']:.4f} | val_f1={val_metrics['f1']:.4f}"
        )

        if val_metrics["f1"] > best_val_f1:
            best_val_f1 = val_metrics["f1"]
            best_epoch = epoch
            best_val_metrics = val_metrics
            epochs_no_improve = 0

            ensure_dir(best_model_dir)
            model.save_pretrained(best_model_dir)
            tokenizer.save_pretrained(best_model_dir)
            save_json(
                os.path.join(config.output_dir, "best_val_metrics.json"),
                {
                    "epoch": epoch,
                    "val_loss": val_metrics["loss"],
                    "val_acc": val_metrics["accuracy"],
                    "val_f1": val_metrics["f1"],
                    "val_precision": val_metrics["precision"],
                    "val_recall": val_metrics["recall"],
                },
            )
            print(f"  → Новый лучший val_f1={best_val_f1:.4f}, чекпоинт сохранён")
        else:
            epochs_no_improve += 1
            print(f"  → Нет улучшения ({epochs_no_improve}/{config.patience})")
            if epochs_no_improve >= config.patience:
                print("Early stopping.")
                break

    print(f"\nЛучшая эпоха: {best_epoch}, best_val_f1={best_val_f1:.4f}")

    # Загружаем лучший чекпоинт для финальной оценки на тесте
    best_model = AutoModelForSequenceClassification.from_pretrained(best_model_dir).to(
        config.device
    )
    test_metrics = evaluate(best_model, test_loader, config.device)

    print("\n=== Test metrics ===")
    print(f"test_loss:      {test_metrics['loss']:.4f}")
    print(f"test_acc:       {test_metrics['accuracy']:.4f}")
    print(f"test_f1:        {test_metrics['f1']:.4f}")
    print(f"test_precision: {test_metrics['precision']:.4f}")
    print(f"test_recall:    {test_metrics['recall']:.4f}")
    print("\nClassification report:")
    print(
        classification_report(test_metrics["y_true"], test_metrics["y_pred"], digits=4)
    )

    save_training_artifacts(config, history, best_epoch, best_val_metrics, test_metrics)

    return {
        "config": asdict(config),
        "history": history,
        "best_epoch": best_epoch,
        "best_val_metrics": best_val_metrics,
        "test_metrics": {
            "accuracy": test_metrics["accuracy"],
            "f1": test_metrics["f1"],
            "precision": test_metrics["precision"],
            "recall": test_metrics["recall"],
        },
    }

## Инференс

Утилиты для загрузки обученной модели и предсказания на новых текстах.  
`predict_texts` работает **батчами** — значительно быстрее, чем посимвольно по одному тексту.

In [ ]:
def load_inference_artifacts(model_dir="../../models/rubert/best_model", device=None):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir).to(device)
    model.eval()
    return tokenizer, model, device


@torch.no_grad()
def predict_texts(texts, tokenizer, model, device, max_length=256, batch_size=32):
    """Батчевый инференс. Возвращает список словарей с label, fake_proba, real_proba."""
    results = []
    for i in range(0, len(texts), batch_size):
        batch_texts = [str(t) for t in texts[i : i + batch_size]]
        encoded = tokenizer(
            batch_texts,
            truncation=True,
            padding=True,
            max_length=max_length,
            return_tensors="pt",
        )
        encoded = {k: v.to(device) for k, v in encoded.items()}
        logits = model(**encoded).logits
        probs = torch.softmax(logits, dim=1).cpu().numpy()
        preds = probs.argmax(axis=1)

        for pred, prob in zip(preds, probs):
            results.append(
                {
                    "label": int(pred),
                    "label_name": "real" if pred == 1 else "fake",
                    "fake_proba": float(prob[0]),
                    "real_proba": float(prob[1]),
                }
            )
    return results


@torch.no_grad()
def predict_text(text, tokenizer, model, device, max_length=256):
    """Инференс одного текста."""
    return predict_texts([text], tokenizer, model, device, max_length, batch_size=1)[0]

## Запуск обучения

Создаём конфиг и запускаем обучение. Для быстрой проверки можно уменьшить `epochs` или `batch_size`.

In [ ]:
config = Config()
results = train_rubert(config)